In [1]:
#!pip install control

In [1]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42  

# Linear Model

 - $\hat{y}_1 = h(x_1)$ 
     * $\frac{d \hat{y}_1}{d x_0} = h'(x)\frac{d x_1}{d x_0}$
     * $\frac{d \hat{y}_1}{d w} = h'(x)\frac{d x_1}{d w} =  h'(x)(\frac{d x_1}{d x_0} \frac{d x_0}{d w} +\frac{\partial x_1}{\partial w}) // \frac{d x_0}{d w} = 0$
     
  - $\hat{y}_{t-1} = h(x_{t-1})$ 
     * $\frac{d \hat{y}_{t-1}}{d x_0} = h'(x)\frac{d x_{t-1}}{d x_0} = h'(x) \frac{d x_{t-1}}{d x_{t-2}}\frac{d x_{t-2}}{d x_0}$
     * $\frac{d \hat{y}_{t-1}}{d w} = h'(x)\frac{d x_{t-1}}{d w} = h'(x)(\frac{d x_{t-1}}{d x_{t-2}}\frac{d x_{t-2}}{d_w} + \frac{d x_{t-1}}{d_w})$
     
  - $\hat{y}_{t} = h(x_{t})$ 
     * $\frac{d \hat{y}_{t}}{d x_0} = h'(x)\frac{d x_{t}}{d x_0} = h'(x) \frac{d x_{t}}{d x_{t-1}}\frac{d x_{t-1}}{d x_0}$
     * $\frac{d \hat{y}_{t}}{d w} = h'(x)\frac{d x_{t}}{d w} = h'(x)(\frac{d x_{t}}{d x_{t-1}}\frac{d x_{t-1}}{d_w} + \frac{d x_t}{d_w})$    

In [2]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def save2file(line, cell):
    'save python code block to a file'
    with open(line, 'wt') as fd:
        fd.write(cell)

In [3]:
%%save2file tmp.py
import time
import numpy as np
import control as ct
import matplotlib.pyplot as plt
import copy

from tqdm import tqdm
from tqdm.notebook import tnrange

import pickle

import torch
from torch.optim import AdamW,Adam,SGD,NAdam
from scipy.optimize import basinhopping
from scipy.optimize import minimize as sciminimize
from scipy.stats import chi2
from scipy.special import expit
import scipy

from adaptivesysid.simu import *
from adaptivesysid.model_base import *
from adaptivesysid.cruisecontrol import *
from adaptivesysid.pendulum import *
from adaptivesysid.watertank import *
from adaptivesysid.torch_gradient_descent import *
from adaptivesysid.para import *
from adaptivesysid.model_jit import *
import sys
import os
import torch.multiprocessing as multiprocessing
   
    
def task(seed,U,U_test,model_eval,model_est,estimator,evaluator,simulator,repeat,x_init0,theta0,invC0,gamma0,invS0,true_theta,block_size,_rng,output,Q,a,dt,result_dict):
    def generate_data(u,seed):
        try:
            x,y = simulator(u,Q,seed=seed,dt=dt)
        except:
            x,y = simulator(u,Q,seed=seed,dt=dt)

        if len(y.shape)==1:
             y = y.reshape(1,-1)
        if len(x.shape)==1:
            x = x.reshape(1,-1)
        u = u.reshape(1,-1)

        return x,u,y 

    #seed = res[0]
    #us = res[1]
    if os.path.isfile(output):
        with open(output,'rb') as f:
            U_done = pickle.load(f)
    else:
        U_done = {}
        
    if str(seed) in U_done:
        u = U_done[str(seed)]
    else:
        u = U(seed)
       
    x,u,y = generate_data(u,seed)
        
    seed_test = int(seed-10)
    x_test,u_test,y_test = generate_data(U_test(seed_test),seed_test)
        
    mses = []
    crbs = []
    thetaerrs = []
    stateerrs = []
    powers = []
    theta_est = theta0.copy()
    invC_est = invC0.copy()
    C_prev = np.linalg.inv(invC0)
    invS_est = invS0.copy()
    gamma_est = gamma0.copy()
    x_init_est = x_init0.copy()
    
    ns = []
    t = 1

    
    for i in _rng:
        if i%block_size>0:
            continue
                
        ns.append(i)
            
        #Step 0. Powers
        powers.append(np.mean(u[:i]**2)/(a**2))
            
        #Step 1. Evaluate
        crb,mse,C_prev = evaluator(model_eval,block_size,x[:,:i],true_theta,C_prev,y[:,:i],u[:,:i],u_test,y_test)
        crbs.append(crb)
        mses.append(mse)
            
        #Step 2. Append estimation errors
        model_est.gamma2 = gamma_est
        model_est.invGamma = torch.diag(1/torch.tensor(gamma_est).flatten())
        model_est.invGamma_np = np.diag(1/gamma_est.flatten())
        print(model_est.invGamma_np,gamma_est)
        x_init_est,theta_est,invC_est,gamma_est,invS_est = estimator(model_est,block_size,x_init_est,theta_est,invC_est,gamma_est,invS_est,y[:,:i],u[:,:i],t) 

    
        t+=1
        thetaerr = (((theta_est.flatten()-np.array(true_theta).flatten())**2)/(true_theta.flatten()**2)).sum()
        
        thetaerrs.append(thetaerr)
        
    result_dict[str(seed)] = (seed,mses,crbs,thetaerrs,powers,u,y,x,ns)
    

def monte_carlo_ss_parallel(U,U_test,model_eval,model_est,estimator,evaluator,simulator,repeat,x_init0,theta0,invC0,gamma0,invS0,true_theta,block_size,_rng,output,output_pk,Q,PROCESSES = 4,a=5,dt=1):
    
    pred_errs = []
    theta_crbs = []
    theta_mses = []
    powerss = []
    uss = []
    yss = []
    xss = []
    nss = []
    seeds = [seed+25 for seed in range(repeat)]
    
    multiprocessing.freeze_support()
    processes = []
    result_dict = {}
    
   
    
    for i in range(int(np.ceil(len(seeds)/PROCESSES))):
        manager = multiprocessing.Manager()
        return_dict = manager.dict()
        jobs = []
        
        for j in range(min(PROCESSES,len(seeds)-i*PROCESSES)):
            seed = seeds[i*PROCESSES+j]
            print(seed)
            p = multiprocessing.Process(target=task, args=(seed,U,U_test,copy.deepcopy(model_eval),copy.deepcopy(model_est),estimator
                                                           ,evaluator,simulator,repeat,copy.deepcopy(x_init0),copy.deepcopy(theta0),copy.deepcopy(invC0)
                                                           ,copy.deepcopy(gamma0),copy.deepcopy(invS0),copy.deepcopy(true_theta),block_size,copy.deepcopy(_rng),output,Q,a,dt,return_dict))
            jobs.append(p)
            p.start()

        for proc in jobs:
            proc.join()
       
        if os.path.isfile(output):
            with open(output,'rb') as f:
                U_done = pickle.load(f)
        else:
            U_done = {}

        for j in range(min(PROCESSES,len(seeds)-i*PROCESSES)):
            seed = seeds[i*PROCESSES+j]
            seed,mses,crbs,thetaerrs,powers,u,y,x,ns = return_dict[str(seed)]
            U_done[str(seed)] = u.flatten()
            pred_errs.append(mses)
            theta_crbs.append(crbs)
            theta_mses.append(thetaerrs)
            powerss.append(powers)
            uss.append(u)
            yss.append(y)
            xss.append(x)
            nss.append(ns)


        with open(output_pk,'wb') as f:
            pickle.dump((np.array(pred_errs),np.array(theta_crbs),np.array(theta_mses),np.array(powerss),np.array(uss),np.array(yss),np.array(xss),np.array(nss)),f)

        with open(output,'wb') as f:
            pickle.dump(U_done,f)
        time.sleep(1)
    print('done executing')


def estimatorS300(model,block_size,x_init0,theta0,invC0,gamma0,invS0,ys,us,t):
   
    N = ys.shape[1]
    
    if t==1:
        lr = 1e-2
        n_iter = 20000
        x_init1,x_init0,theta,invC = recursive_mle(model,ys,us,block_size,theta0,x_init0,invC0,np.eye(1)*gamma0,lr=lr,n_iter=n_iter)
        print('first try',theta)
        x_init1,x_init0,theta,invC = recursive_mle_opt(model,ys,us,block_size,theta,x_init0,invC0,0.1,600)
        gamma1,invS1 = recursive_mle_sigma(model,ys,us,block_size,theta,x_init0,invC,gamma0,invS0,n_iter=100)
    else:
        x_init1,x_init0,theta,invC = recursive_mle_opt(model,ys,us,block_size,theta0,x_init0,invC0,0.1,600)
        gamma1,invS1 = recursive_mle_sigma(model,ys,us,block_size,theta,x_init0,invC,gamma0,invS0,n_iter=100)
   
    print('estimation theta',theta,np.linalg.inv(invC))
    return x_init1,theta,invC,gamma1,invS1

   

def estimatorS600(model,block_size,x_init0,theta0,invC0,gamma0,invS0,ys,us,t):
    N = ys.shape[1]

    
    if t==1:
        print(gamma0,invC0,invS0,model.gamma2)
        lr=3e-1
        n_iter = 1000
        
        x_init1,x_init0,theta,invC = recursive_mle(model,ys,us,block_size,theta0,x_init0,invC0,np.eye(1)*gamma0,lr=lr,n_iter=n_iter)
        print('first try',theta)
        
        lr = 1e-2
        n_iter = 1000
        x_init1,x_init0,theta,invC = recursive_mle(model,ys,us,block_size,theta,x_init0,invC0,np.eye(1)*gamma0,lr=lr,n_iter=n_iter)
        print('second try',theta)
        
        print(ys,us)
        x_init1,x_init0,theta,invC = recursive_mle_opt(model,ys,us,block_size,theta,x_init0,invC0,0.1,600)
        gamma1,invS1 = recursive_mle_sigma(model,ys,us,block_size,theta,x_init0,invC,gamma0,invS0,n_iter=200)
    else:
        x_init1,x_init0,theta,invC = recursive_mle_opt(model,ys,us,block_size,theta0,x_init0,invC0,0.1,600)
        gamma1,invS1 = recursive_mle_sigma(model,ys,us,block_size,theta,x_init0,invC,gamma0,invS0,n_iter=200)
    
    
    print('estimation theta',theta,np.linalg.inv(invC),'x_init',x_init0,'x_prop',x_init1)
    return x_init1,theta,invC,gamma1,invS1

def estimatorS601(model,block_size,x_init0,theta0,invC0,gamma0,invS0,ys,us,t):
    N = ys.shape[1]
    print('N=',N,block_size)

    x_init,theta,invC = ekf_opt(model_est,ys[:,[-1]],us[:,[-2]],theta0,x_init0,invC0)
    
    gamma1,invS1 = recursive_mle_sigma(model,ys,us,1,theta,x_init0,invC,gamma0,invS0,n_iter=200)

    
    print('estimation theta',theta)
    return x_init,theta,invC,gamma1,invS1

    
def estimatorS400(model,block_size,x_init0,theta0,invC0,gamma0,invS0,ys,us,t):
    N = ys.shape[1]
    
    if t==1:
        lr = 1
        n_iter = 1000
        x_init1,x_init0,theta,invC = recursive_mle(model,ys,us,block_size,theta0,x_init0,invC0,np.diag(gamma0.flatten()),lr=lr,n_iter=n_iter)
        print('first try',theta)
        lr = 1e-1
        n_iter = 1000
        x_init1,x_init0,theta,invC = recursive_mle(model,ys,us,block_size,theta,x_init0,invC0,np.diag(gamma0.flatten()),lr=lr,n_iter=n_iter)
        print('second try',theta)
        
        x_init,x_init0,theta,invC = recursive_mle_opt(model,ys,us,block_size,theta,x_init0,invC0,0.1,600)
        gamma1,invS1 = recursive_mle_sigma(model,ys,us,block_size,theta,x_init0,invC,gamma0,invS0,n_iter=200)
        
    else:
        x_init1,x_init0,theta,invC = recursive_mle_opt(model,ys,us,block_size,theta0,x_init0,invC0,0.1,800)
        gamma1,invS1 = recursive_mle_sigma(model,ys,us,block_size,theta,x_init0,invC,gamma0,invS0,n_iter=200)

            
    print('estimation theta',theta,invC,np.linalg.pinv(invC))
    return x_init1,theta,invC,gamma1,invS1

L = 60
lambda_true=0.01
Q = np.array([lambda_true**2])
torch.set_grad_enabled(True)
torch.set_default_dtype(torch.double)



T = 64
amplitude = 10 
dt = 0.1
simulator = simulate_nonlinear19
estimator = estimatorS600
gamma2 = np.ones((1,1))*(lambda_true**2)
QS600 = np.eye(1)*gamma2
model_eval = StateSpacePendulumModel([-amplitude,amplitude],gamma2,dt)
true_theta = np.array([[-24],[1]])
theta0 = np.ones((2,1))
invC0 = np.eye(4)*1e-4
gamma0 = np.ones((1,1))*(lambda_true**2)
model_est = StateSpacePendulumModel([-amplitude,amplitude],gamma0,dt)
invS0 = np.eye(1)*1e14
x_init0 = np.ones((2,1))
block_size = 7
_rngS600 = list(range(block_size,T,1))

def US600_T(seed):
    return np.roll(scaled_PRBS_RA(seed,100,scale=amplitude,dt=dt),9)

def US601(seed):
    return np.roll(scaled_PRBS(100,scale=amplitude,seed=seed,dt=dt),9)

def US600014(seed):
    K = 6
    model = BarrierPendulumModelJ([-amplitude,amplitude],gamma0,dt,np.array([[-np.pi/4,np.pi/4]]),K,200.)
    model.init_numpy()
    simulator = simulate_nonlinear19
    estimator = estimatorS600
    us = US601(seed)[:block_size]
    return generate_uk_ss(model,estimator,T,K,us,theta0,x_init0,invC0,gamma0,invS0,simulator,QS600,block_size,weight='full',seed=seed,dt=dt,lr_desc=0.01,n_iter=3000)




if __name__ == '__main__':
   
    __spec__ = "ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>)"
    
   
     
    monte_carlo_ss_parallel(US601,US600_T,model_eval,model_est,estimator,evaluatorSS,simulator,36,x_init0,theta0,invC0,gamma0,invS0,true_theta,block_size,_rngS600,'plotsdata/S601_U.pk','plotsdata/S601.pk',QS600,PROCESSES = 8,a=amplitude,dt=dt)
    monte_carlo_ss_parallel(US600014,US600_T,model_eval,model_est,estimator,evaluatorSS,simulator,100,x_init0,theta0,invC0,gamma0,invS0,true_theta,block_size,_rngS600,'plotsdata/S600014_U.pk','plotsdata/S600014.pk',QS600,PROCESSES = 7,a=amplitude,dt=dt)
    

In [ ]:
%run tmp.py

In [153]:
def joint_ekf(model,ys,us,theta0,x_init0,P0,R0):
    dim_theta = theta0.shape[0]
    thetax = np.vstack([theta0,x_init0])
    eye_zero = np.hstack([np.eye(dim_theta),np.zeros((dim_theta,x_init0.shape[0]))])
    
    x_ = x_init0.copy()
    theta_ = theta0.copy()
    P = P0.copy()
    R = R0.copy()
    I = np.eye(P.shape[0])
    H = model.H3
    eps = 1e-3
    thetas = []
    for i in range(us.shape[1]-1):
        u = us[:,[i]]
        
        # Time Update
        x_ = model.transition_np(theta_,x_,u)
      
        thetax = np.vstack([theta_,x_])
        dx_theta = model.dxtheta(torch.tensor(theta_),torch.tensor(x_),torch.tensor(u))
        dx_x = model.dxx(torch.tensor(theta_),torch.tensor(x_),torch.tensor(u))
        
        
        F = np.vstack([eye_zero,np.hstack([dx_theta,dx_x])])
        
        P = F@P@F.T
        
        # Measurement Update
        y = ys[:,[i+1]]
        yhat = H@thetax
        v = y-yhat
        S = H@P@H.T+R
        K = P@H.T@np.linalg.inv(S)
                                
        
        thetax = thetax + K@v
        P = (I - K@H)@P@((I - K@H).T) + K@R@K.T
        
        ## R update
        #R_ = v@v.T-H@P@H.T
        #beta = 1/(i+1)
        #diag_R = np.maximum(np.diag(0.5*(R_+R_.T)),eps)
        #R = (1-beta)*R + beta*np.diag(diag_R)
        
        theta_ = thetax[:dim_theta,:]
        x_ = thetax[dim_theta:,:]
        
        thetas.append(theta_)
    
    return np.array(thetas)